# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nando135/flyrank_ml_1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Baseline rule: Prioritize content pages for review when they have high search visibility but weak response. Pages with many impressions but relatively low clicks or weak engagement should receive a higher baseline score because they may represent an opportunity for improvement.
Reason code: high_visibility_low_response
Action label: review_content_and_search_snippet

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')"
)

print("DuckDB connection ready")

DuckDB connection ready


In [31]:
baseline_query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
"""

baseline_df = con.sql(baseline_query).df()

print("Rows:", len(baseline_df))
display(baseline_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 364347


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,True,True,True,True,5,0,27,...,0,0,0,0,0,0,0,0,0,2026-03
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,True,True,True,True,39,0,221,...,1,0,0,0,1,0,0,0,0,2026-03
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,True,True,True,True,179,0,923,...,1,0,0,1,0,0,0,0,0,2026-03
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,True,True,True,True,72,0,554,...,1,0,0,1,0,0,0,0,0,2026-03
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,True,True,True,True,3282,1,20243,...,0,0,0,0,0,0,0,0,0,2026-03


In [32]:
page_df = (
    baseline_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        sessions=("ga4_sessions", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        total_engagement_sec=("ga4_total_engagement_sec", "sum")
    )
)

page_df["ctr"] = (
    page_df["clicks"] / page_df["impressions"]
).fillna(0)

page_df["engagement_rate"] = (
    page_df["engaged_sessions"] / page_df["sessions"]
).fillna(0)

print("Pages:", len(page_df))
display(page_df.head())

Pages: 63856


,client_hash_id,content_hash_id,impressions,clicks,sessions,engaged_sessions,total_engagement_sec,ctr,engagement_rate
0,client_08d2847f24cf89c1,content_0071174ee1f89d8a,22,2,2,0,0,0.090909,0.0
1,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,186,7,7,0,2,0.037634,0.0
2,client_08d2847f24cf89c1,content_0f0c4d121b9d4aa5,8,1,4,0,0,0.125000,0.0
3,client_08d2847f24cf89c1,content_12444e383a53d45f,3,0,1,0,0,0.000000,0.0
4,client_08d2847f24cf89c1,content_145bbdb1286836a0,2,0,1,0,0,0.000000,0.0


In [33]:
import pandas as pd

page_df["impression_tier"] = pd.qcut(
    page_df["impressions"],
    q=4,
    labels=["low", "medium", "high", "very_high"],
    duplicates="drop"
)

signal_1 = (
    page_df
    .groupby("impression_tier", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum")
    )
    .reset_index()
)

signal_1["ctr"] = (
    signal_1["total_clicks"] / signal_1["total_impressions"]
)

display(signal_1)

,impression_tier,n,total_impressions,total_clicks,ctr
0,low,16268,103358,4888,0.047292
1,medium,15665,790098,12102,0.015317
2,high,15966,4858897,39951,0.008222
3,very_high,15957,79584367,338263,0.004250


In [34]:
signal_1["impression_growth"] = (
    signal_1["total_impressions"]
    / signal_1["total_impressions"].shift(1)
)

signal_1["click_growth"] = (
    signal_1["total_clicks"]
    / signal_1["total_clicks"].shift(1)
)

signal_1["response_growth_ratio"] = (
    signal_1["click_growth"]
    / signal_1["impression_growth"]
)

display(signal_1)

,impression_tier,n,total_impressions,total_clicks,ctr,impression_growth,click_growth,response_growth_ratio
0,low,16268,103358,4888,0.047292,NaN,NaN,NaN
1,medium,15665,790098,12102,0.015317,7.644285,2.475859,0.323884
2,high,15966,4858897,39951,0.008222,6.149740,3.301190,0.536802
3,very_high,15957,79584367,338263,0.004250,16.379101,8.466947,0.516936


In [35]:
signal_1["impression_increase"] = (
    signal_1["total_impressions"].diff()
)

signal_1["click_increase"] = (
    signal_1["total_clicks"].diff()
)

signal_1["clicks_per_extra_impression"] = (
    signal_1["click_increase"]
    / signal_1["impression_increase"]
)

signal_1["extra_impressions_per_click"] = (
    signal_1["impression_increase"]
    / signal_1["click_increase"]
)

display(signal_1)

,impression_tier,n,total_impressions,total_clicks,ctr,impression_growth,click_growth,response_growth_ratio,impression_increase,click_increase,clicks_per_extra_impression,extra_impressions_per_click
0,low,16268,103358,4888,0.047292,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,medium,15665,790098,12102,0.015317,7.644285,2.475859,0.323884,686740.0,7214.0,0.010505,95.195453
2,high,15966,4858897,39951,0.008222,6.149740,3.301190,0.536802,4068799.0,27849.0,0.006845,146.102158
3,very_high,15957,79584367,338263,0.004250,16.379101,8.466947,0.516936,74725470.0,298312.0,0.003992,250.494348


In [36]:
signal_2 = (
    page_df
    .groupby("impression_tier", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        total_sessions=("sessions", "sum"),
        total_engaged_sessions=("engaged_sessions", "sum")
    )
    .reset_index()
)

signal_2["engagement_rate"] = (
    signal_2["total_engaged_sessions"]
    / signal_2["total_sessions"]
)

display(signal_2)

,impression_tier,n,total_sessions,total_engaged_sessions,engagement_rate
0,low,16268,29875,817,0.027347
1,medium,15665,82642,1490,0.018030
2,high,15966,220927,3702,0.016757
3,very_high,15957,905311,21964,0.024261


In [37]:
signal_2["session_increase"] = (
    signal_2["total_sessions"].diff()
)

signal_2["engaged_session_increase"] = (
    signal_2["total_engaged_sessions"].diff()
)

signal_2["engaged_sessions_per_extra_session"] = (
    signal_2["engaged_session_increase"]
    / signal_2["session_increase"]
)

signal_2["extra_sessions_per_engaged_session"] = (
    signal_2["session_increase"]
    / signal_2["engaged_session_increase"]
)

display(signal_2)

,impression_tier,n,total_sessions,total_engaged_sessions,engagement_rate,session_increase,engaged_session_increase,engaged_sessions_per_extra_session,extra_sessions_per_engaged_session
0,low,16268,29875,817,0.027347,NaN,NaN,NaN,NaN
1,medium,15665,82642,1490,0.018030,52767.0,673.0,0.012754,78.405646
2,high,15966,220927,3702,0.016757,138285.0,2212.0,0.015996,62.515823
3,very_high,15957,905311,21964,0.024261,684384.0,18262.0,0.026684,37.475851


In [45]:
page_df["visibility_score"] = (
    page_df["impressions"].rank(pct=True)
)

page_df["low_ctr_score"] = (
    1 - page_df["ctr"].rank(pct=True)
)

page_df["low_engagement_score"] = (
    1 - page_df["engagement_rate"].rank(pct=True)
)

display(
    page_df[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "engagement_rate",
            "visibility_score",
            "low_ctr_score",
            "low_engagement_score"
        ]
    ].head()
)

,client_hash_id,content_hash_id,impressions,ctr,engagement_rate,visibility_score,low_ctr_score,low_engagement_score
0,client_08d2847f24cf89c1,content_0071174ee1f89d8a,22,0.090909,0.0,0.283278,0.056220,0.599842
1,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,186,0.037634,0.0,0.579867,0.111861,0.599842
2,client_08d2847f24cf89c1,content_0f0c4d121b9d4aa5,8,0.125000,0.0,0.168535,0.044052,0.599842
3,client_08d2847f24cf89c1,content_12444e383a53d45f,3,0.000000,0.0,0.087423,0.808804,0.599842
4,client_08d2847f24cf89c1,content_145bbdb1286836a0,2,0.000000,0.0,0.060762,0.808804,0.599842


In [46]:
page_df["visibility_score"] = (
    page_df["impressions"].rank(pct=True)
)

page_df["low_ctr_score"] = (
    1 - page_df["ctr"].rank(pct=True)
)

page_df["low_engagement_score"] = (
    1 - page_df["engagement_rate"].rank(pct=True)
)

page_df["baseline_score"] = (
    0.40 * page_df["visibility_score"]
    + 0.40 * page_df["low_ctr_score"]
    + 0.20 * page_df["low_engagement_score"]
)

page_df["reason_code"] = "high_visibility_low_response"
page_df["action"] = "review_content_and_search_snippet"

ranked_queue = (
    page_df
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

display(
    ranked_queue[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr",
            "sessions",
            "engagement_rate",
            "visibility_score",
            "low_ctr_score",
            "low_engagement_score",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

,client_hash_id,content_hash_id,impressions,clicks,ctr,sessions,engagement_rate,visibility_score,low_ctr_score,low_engagement_score,baseline_score,reason_code,action
0,client_23a62021009f63c4,content_559cdd76da9306de,43776,0,0.0,169,0.0,0.997964,0.808804,0.599842,0.842676,high_visibility_low_response,review_content_and_search_snippet
1,client_23a62021009f63c4,content_bf078007df823490,37262,0,0.0,12,0.0,0.997040,0.808804,0.599842,0.842306,high_visibility_low_response,review_content_and_search_snippet
2,client_23a62021009f63c4,content_1162dc8495e06dfb,16929,0,0.0,13,0.0,0.986861,0.808804,0.599842,0.838234,high_visibility_low_response,review_content_and_search_snippet
3,client_23a62021009f63c4,content_d2def933ed902af2,14118,0,0.0,47,0.0,0.982414,0.808804,0.599842,0.836455,high_visibility_low_response,review_content_and_search_snippet
4,client_23a62021009f63c4,content_b47e98f291c7e6cb,11229,0,0.0,42,0.0,0.975727,0.808804,0.599842,0.833781,high_visibility_low_response,review_content_and_search_snippet
5,client_23a62021009f63c4,content_bd63db2d0757e760,11091,0,0.0,396,0.0,0.975429,0.808804,0.599842,0.833662,high_visibility_low_response,review_content_and_search_snippet
6,client_23a62021009f63c4,content_c367b0ca57f3559b,10530,0,0.0,10,0.0,0.973753,0.808804,0.599842,0.832991,high_visibility_low_response,review_content_and_search_snippet
7,client_23a62021009f63c4,content_5a10cbcbb9ec8ef6,9962,0,0.0,14,0.0,0.971749,0.808804,0.599842,0.832190,high_visibility_low_response,review_content_and_search_snippet
8,client_23a62021009f63c4,content_4ae1505344734f99,8758,0,0.0,206,0.0,0.966910,0.808804,0.599842,0.830254,high_visibility_low_response,review_content_and_search_snippet
9,client_23a62021009f63c4,content_1ff3c48911f11e70,8581,0,0.0,20,0.0,0.965861,0.808804,0.599842,0.829834,high_visibility_low_response,review_content_and_search_snippet


In [47]:
import os

os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved: work/outputs/baseline_action_score.csv")

Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_review = ranked_queue.head(20)[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "sessions",
        "engaged_sessions",
        "engagement_rate",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

display(top20_review)

,client_hash_id,content_hash_id,impressions,clicks,ctr,sessions,engaged_sessions,engagement_rate,baseline_score,reason_code,action
0,client_23a62021009f63c4,content_559cdd76da9306de,43776,0,0.0,169,0,0.0,0.842676,high_visibility_low_response,review_content_and_search_snippet
1,client_23a62021009f63c4,content_bf078007df823490,37262,0,0.0,12,0,0.0,0.842306,high_visibility_low_response,review_content_and_search_snippet
2,client_23a62021009f63c4,content_1162dc8495e06dfb,16929,0,0.0,13,0,0.0,0.838234,high_visibility_low_response,review_content_and_search_snippet
3,client_23a62021009f63c4,content_d2def933ed902af2,14118,0,0.0,47,0,0.0,0.836455,high_visibility_low_response,review_content_and_search_snippet
4,client_23a62021009f63c4,content_b47e98f291c7e6cb,11229,0,0.0,42,0,0.0,0.833781,high_visibility_low_response,review_content_and_search_snippet
5,client_23a62021009f63c4,content_bd63db2d0757e760,11091,0,0.0,396,0,0.0,0.833662,high_visibility_low_response,review_content_and_search_snippet
6,client_23a62021009f63c4,content_c367b0ca57f3559b,10530,0,0.0,10,0,0.0,0.832991,high_visibility_low_response,review_content_and_search_snippet
7,client_23a62021009f63c4,content_5a10cbcbb9ec8ef6,9962,0,0.0,14,0,0.0,0.832190,high_visibility_low_response,review_content_and_search_snippet
8,client_23a62021009f63c4,content_4ae1505344734f99,8758,0,0.0,206,0,0.0,0.830254,high_visibility_low_response,review_content_and_search_snippet
9,client_23a62021009f63c4,content_1ff3c48911f11e70,8581,0,0.0,20,0,0.0,0.829834,high_visibility_low_response,review_content_and_search_snippet


Action: review_content_and_search_snippet — Reason: 43,776 impressions with 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: the zero-response values are caused by client-specific tracking or measurement issues.

Action: review_content_and_search_snippet — Reason: 37,262 impressions with 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: GSC or GA4 reporting for this page is incomplete.

Action: review_content_and_search_snippet — Reason: 16,929 impressions with 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: the page is intentionally visible in search but is not expected to generate clicks.

Action: review_content_and_search_snippet — Reason: 14,118 impressions with 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: the zero-response pattern is caused by attribution or tracking problems.

Action: review_content_and_search_snippet — Reason: 11,229 impressions with 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: the page is appearing for queries that are not relevant to its intended audience.

Action: review_content_and_search_snippet — Reason: 11,091 impressions and 396 sessions, but 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: engaged sessions are not being measured correctly for this page.

Action: review_content_and_search_snippet — Reason: 10,530 impressions with 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: the search appearances are not intended to produce direct clicks.

Action: review_content_and_search_snippet — Reason: 9,962 impressions with 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: the page serves a purpose that is not captured by search clicks or engagement.

Action: review_content_and_search_snippet — Reason: 8,758 impressions and 206 sessions, but 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: GA4 engagement tracking is incomplete or misconfigured.

Action: review_content_and_search_snippet — Reason: 8,581 impressions with 0 CTR and 0 engagement rate. Confidence: Medium-high. Could be wrong if: the page is receiving impressions from broad or irrelevant search queries.

Action: review_content_and_search_snippet — Reason: 8,517 impressions and 166 sessions, but 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: engagement events are not being recorded correctly.

Action: review_content_and_search_snippet — Reason: 8,353 impressions with 0 CTR and 0 engagement rate. Confidence: Medium-high. Could be wrong if: the page is intentionally not optimized for direct search interaction.

Action: review_content_and_search_snippet — Reason: 7,700 impressions with 0 CTR and 0 engagement rate. Confidence: Medium-high. Could be wrong if: the observed zeros come from incomplete measurement.

Action: review_content_and_search_snippet — Reason: 7,700 impressions with 0 CTR and 0 engagement rate. Confidence: Medium-high. Could be wrong if: the page appears for queries that do not match the page's intended purpose.

Action: review_content_and_search_snippet — Reason: 7,486 impressions and 157 sessions, but 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: engaged-session tracking is missing for this content.

Action: review_content_and_search_snippet — Reason: 6,835 impressions and 81 sessions, but 0 CTR and 0 engagement rate. Confidence: Medium-high. Could be wrong if: the page's value is delivered through behavior not represented by these metrics.

Action: review_content_and_search_snippet — Reason: 6,824 impressions and 115 sessions, but 0 CTR and 0 engagement rate. Confidence: Medium-high. Could be wrong if: the zero engagement reflects tracking limitations rather than actual user behavior.

Action: review_content_and_search_snippet — Reason: 6,756 impressions with 0 CTR and 0 engagement rate. Confidence: Medium-high. Could be wrong if: the page is receiving low-intent or irrelevant search exposure.

Action: review_content_and_search_snippet — Reason: 6,596 impressions with 0 CTR and 0 engagement rate. Confidence: Medium. Could be wrong if: this page's search visibility does not represent a realistic click opportunity.

Action: review_content_and_search_snippet — Reason: 6,329 impressions and 363 sessions, but 0 CTR and 0 engagement rate. Confidence: High. Could be wrong if: engagement events are not being captured correctly for this page.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [50]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_pick_check = top20_review[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "sessions",
        "engaged_sessions",
        "engagement_rate",
        "baseline_score"
    ]
].copy()

display(
    weak_pick_check.sort_values(
        ["impressions", "sessions"],
        ascending=True
    )
)

,client_hash_id,content_hash_id,impressions,clicks,ctr,sessions,engaged_sessions,engagement_rate,baseline_score
19,client_23a62021009f63c4,content_066bb7aeff9aeea8,6329,0,0.0,363,0,0.0,0.823877
18,client_23a62021009f63c4,content_7c08b008c301111e,6596,0,0.0,30,0,0.0,0.824782
17,client_23a62021009f63c4,content_a27b382f00aa75c6,6756,0,0.0,40,0,0.0,0.825387
16,client_23a62021009f63c4,content_fe8baba849843607,6824,0,0.0,115,0,0.0,0.825568
15,client_23a62021009f63c4,content_c055d2923bce1641,6835,0,0.0,81,0,0.0,0.825600
14,client_23a62021009f63c4,content_f1d47f46593eea32,7486,0,0.0,157,0,0.0,0.827404
12,client_23a62021009f63c4,content_a75e70dd8593bffc,7700,0,0.0,13,0,0.0,0.827924
13,client_23a62021009f63c4,content_1260444b6f9540b9,7700,0,0.0,42,0,0.0,0.827924
11,client_23a62021009f63c4,content_70d9d7d2c814b431,8353,0,0.0,34,0,0.0,0.829371
10,client_23a62021009f63c4,content_7117522ee4db5c58,8517,0,0.0,166,0,0.0,0.829697


The Top-20 do not appear weak because of low search volume, since all have more than 6,000 impressions. However, some picks have very low session counts relative to their impressions, so these should be treated cautiously. This could reflect differences between search exposure and actual site visits, or possible measurement limitations. No future-window information or product-generated flags were used in the baseline; the score only uses March 2026 observed impressions, CTR, and engagement metrics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.